In [5]:
import torch
import torch.nn as nn 
import math

# Implememnting transformer from scratch:


In [6]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self,d_model:int,h:int,dropout:float):
        super().__init__()
        self.d_model=d_model
        assert d_model%h==0 ,"d_model is not divisible by h"
        self.d_k=d_model/h
        self.w_q=nn.Linear(d_model,d_model)
        self.w_k=nn.Linear(d_model,d_model)
        self.w_v=nn.Linear(d_model,d_model)
        
        self.w_o=nn.Linear(d_model,d_model)
        self.dropout=nn.Dropout(dropout)
    @staticmethod
    def attention(query,key,value,mask,dropout:nn.Dropout):
        d_k=query.shape[-1]

        attention_scores=(query@key.transpose(-2,-1))/math.sqrt(d_K)
        if mask is not None:
            attention_score.masked_fill(mask==0,-1e9)
        attention_scores=attention_scores.softmax(dim=-1) #(batch,h,seq,seq)
        if dropout is not None:
            attention_scores=dropout(attention_scores)
        return (attention_scores@value),attention_scores
        
        
            
        

    def forward(self,q,k,v,mask):
        query=self.w_q(q) # (batch,seq,d_model)-->batch,seq,d_model
        key=self.w_k(k) # (batch,seq,d_model)-->batch,seq,d_model
        value=self.w_v(v) # (batch,seq,d_model)-->batch,seq,d_model

        # (batch,seq,d_model)-->(batch,seq,h,d_k)---> (batch,h,seq,d_k)
        

        query=query.view(query.shape[0],query.shape[1],self.h,self.d_k).transpose(1,2)
        key=key.view(key,shape[0],key.shape[1],self.h,self.d_k).transpose(1,2)
        value=value.view(value.shape[0],value.shape[1],self.h,self.d_k).transpose(1,2)
        x,self.attention_scores=MultiHeadAttentionBlock.attention(query,key,value,mask,self.dropout)
        # (batch,h,seq,d_k)-->(batch,seq,h,d_k)-->batch,seq,d_model

        x=x.transpose(1,2).contiguous().view(x.shape[0],-1,self.h*self.d_k)

        return self.w_o(x)
        
        
        
        
        
        
        
        
    
    

In [7]:
print(nn.Parameter(torch.ones(1)))

Parameter containing:
tensor([1.], requires_grad=True)


In [8]:
class LayerNormalization(nn.Module):
    def __init__(self,eps:float=10**-6)->None:
        super().__init__()
        self.eps=eps
        self.alpha=nn.Parameter(torch.ones(1))
        self.bias=nn.Parameter(torch.zeros(1))
    def forwarf(self,x):
        mean=x.mean(dim=-1,keepdim=True)
        std=x.std(dim=-1,keepdim=True)
        return self.alpha*(x-mean)/(std+self.eps)+bias
        
        
        
        

In [9]:
class FeedForwardBlock(nn.Module):
    def __init__(self,d_model:int,d_ff:int,dropout:float)->None:
        super().__init__()
        self.liner_1=nn.Linear(d_model,d_ff)
        self.dropout=nn.dropout(dropout)
        self.linear2=nn.Linear(d_ff,d_model)
    def forward(self,x):
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))



In [10]:
class ResidualConnection(nn.Module):
    def __init__(self,dropout:float):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
        self.norm=LayerNormalization()
    def forward(self,x,sub):
        return x+self.dropout(sub(self.norm(x)))
        
    

In [12]:
class EncoderBlock(nn.Module):
    def __init__(self,self_attention_block:MultiHeadAttentionBlock,feed_forward_block:FeedForwardBlock,dropout:float)->None:
        super().__init__()
        self.self_attention_block=self_attention_block
        self.feed_forward_block=feed_forward_block
        self.residual_connections=nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])
    def forward(self,x,src_mask):
        x=self.residual_connections[0](x, lambda x: self.self_attention_block(x,x,x,src_mask))
        x=self.residual_connections[1](x,self.feed_forward_block)
        return x 
    
        
        
    

In [13]:
class Encoder(nn.Module):
    def __init__(self,layers:nn.ModuleList)->None:
        super().__init__()
        self.layers=layers 
        self.norm=LayerNormalization()
    def forward(self,x ,mask ):
        for layer in self.layers:
            x=layer(x ,mask)
        return self.norm(x)

In [14]:
class DecoderBlock(nn.Module):
    def __init__(self,self_attention_block:MultiHeadAttentionBlock,cross_attention_block:MultiHeadAttentionBlock,feed_forward_block:FeedForwardBlock,dropout:float):
        super().__init__()
        self.self_attention_block=self_attention_block
        self.cross_attention_block=cross_attention_block
        self.feed_forward_block=feed_forwar_block 
        self.residual_connecetions=nn.ModuleList([ResidualConnection(dropout) for _ in range(3)])
    def forward(self,x,encoder_output,src_mask,tgt_mask):
        x=self.residual_connections[0](x,lambda x: self.self_attention_block(x,x,x,tgt_mask))
        x=self.residual_connections[1](x,lambda x: self.cross_attention_block(x,encoder_output,encoder_output,src_mask))
        x=self.residual_connections[2](x,self.feed_forward_block)

        return x 
        

In [16]:
class Decoder(nn.Module):
    def __init__(self ,layers:nn.ModuleList)->None:
        super().__init__()
        self.layers=layers 
        self.norm=LayerNormalization()
    def forward(self,x,encoder_output,src_mask,tgt_mask):
        for layer in self.layers:
            x=layer(x,encoder_output,src_mask,tgt_mask)
        return self.norm(x)
        
            
        
        

In [17]:
class ProjectionLayer(nn.Module):
    def __init__(self,d_model:int,vocab_size:int)->None:
        super().__init__()
        self.proj=nn.Linear(d_model,vocab_size)
    def forward(self,x):
        return torch.log_softmax(self.proj(x),dim=-1)



    

In [20]:
class Transformer(nn.Module):
    def __init__(self,encoder:Encoder,decoder:Decoder,src_embed:InputEmbeddings,tgt_embed:InputEmbeddingd,src_pos:PositionalEmbedding,tgt_pos=PositionalEmbedding,projection_layer:ProjectionLayer)->None:
        super().__init__()
        slef.encoder=encoder
        self.decoder=decoder
        self.src_embed=src_embed
        self.tgt_embed=tgt_embed
        self.src_pos=src_pos
        self.tgt_pos=tgt_pos
        self.projection_layer=projection_layer 
    def encode(self,src,src_mask):
        src=self.src_embed(src)
        src=self.src_pos(src)
        return encoder(src,src_mask)
    def decode(self,encoder_output,src_mask,tgt,tgt_mask):
        tgt=self.tgt_embed(tgt)
        tgt=self.tgt_pos(tgt)
        return self.decoder(tgt,encoder_output,src_mask,tgt_mask)
    def project(self,x):
        return self.projection_laeyr(x)
        
        
        
        
        

SyntaxError: non-default argument follows default argument (3479413049.py, line 2)

In [ ]:
def build_transformer(src_vocab_size:int,tgt_vocab_size:int,src_seq_len:int,tgt_seq_len:int,d_model:int=512,n:int=6,h:int=8,dropout:float=0.1,d_ff:int=2048)->Transformer:
    src_embed=InputEmbeddings(d_model,src_vocab_size)
    tgt_embed=InputEmbeddings(d_model,tgt_vocab_size)
    src_pos=PositionalEmbeddings(d_model,src_seq_len,dropout)
    tgt_pos=PositionalEmbeddings(d_model,tgt_seq_len,dropout)
    for _ in range(n):
        
    
    
    
    
    